In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix, roc_curve, ConfusionMatrixDisplay
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

os.makedirs('models', exist_ok=True)
SEED = 42
print('All libraries loaded ✅')

df = pd.read_csv('data/Student Placement Dataset/train.csv')
print(f'Shape: {df.shape}')
df.head()

TARGET = 'PlacementStatus'   # ← update if your column name differs
print('Target value counts:')
print(df[TARGET].value_counts())

# Drop duplicates
before = len(df)
df.drop_duplicates(inplace=True)
print(f'Removed {before - len(df)} duplicate rows.')

# Encode target
le = LabelEncoder()
df[TARGET] = le.fit_transform(df[TARGET])   # Placed=1, Not Placed=0
print(f'Target classes: {dict(zip(le.classes_, le.transform(le.classes_)))}')

# Drop ID-like columns if any (update list as needed)
drop_cols = [c for c in ['StudentID', 'ID', 'Name'] if c in df.columns]
df.drop(columns=drop_cols, inplace=True, errors='ignore')
print(f'Dropped columns: {drop_cols}')

def safe_norm(series):
    """Min-max normalise a series."""
    mn, mx = series.min(), series.max()
    return (series - mn) / (mx - mn) if mx > mn else series * 0

# ---- Composite feature helpers ----
has = lambda c: c in df.columns

# Career Readiness Score
if all(has(c) for c in ['CGPA','Coding_Skills','Communication_Skills',
                         'Aptitude_Test_Score','Internships','Certifications']):
    df['Career_Readiness_Score'] = (
        0.30 * safe_norm(df['CGPA'])
        + 0.20 * safe_norm(df['Coding_Skills'])
        + 0.15 * safe_norm(df['Communication_Skills'])
        + 0.15 * safe_norm(df['Aptitude_Test_Score'])
        + 0.10 * safe_norm(df['Internships'])
        + 0.10 * safe_norm(df['Certifications'])
    )
    print('Career_Readiness_Score created ✅')

# Academic Strength
if has('CGPA') and has('Aptitude_Test_Score'):
    df['Academic_Strength'] = safe_norm(df['CGPA']) * 0.6 + safe_norm(df['Aptitude_Test_Score']) * 0.4
    print('Academic_Strength created ✅')

# Technical Strength
if all(has(c) for c in ['Coding_Skills','Projects','Certifications']):
    df['Technical_Strength'] = (
        safe_norm(df['Coding_Skills']) * 0.50
        + safe_norm(df['Projects']) * 0.30
        + safe_norm(df['Certifications']) * 0.20
    )
    print('Technical_Strength created ✅')

# Internship Category
if has('Internships'):
    df['Internship_Category'] = pd.cut(
        df['Internships'], bins=[-1, 0, 1, np.inf],
        labels=['None', 'Moderate', 'High']
    ).astype(str)
    print('Internship_Category created ✅')

print(f'\nNew shape: {df.shape}')
df.head(3)

X = df.drop(columns=[TARGET])
y = df[TARGET]

num_features = X.select_dtypes(include=np.number).columns.tolist()
cat_features = X.select_dtypes(include='object').columns.tolist()

print(f'Numerical features ({len(num_features)}): {num_features}')
print(f'Categorical features ({len(cat_features)}): {cat_features}')

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, num_features),
    ('cat', categorical_transformer, cat_features)
], remainder='drop')

print('Preprocessor pipeline built ✅')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

print(f'Train size : {X_train.shape[0]} ({y_train.mean()*100:.1f}% placed)')
print(f'Test size  : {X_test.shape[0]} ({y_test.mean()*100:.1f}% placed)')

models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=SEED, C=1.0
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=10,
        min_samples_split=5, random_state=SEED, n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        use_label_encoder=False, eval_metric='logloss',
        random_state=SEED
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=300, learning_rate=0.05,
        num_leaves=31, random_state=SEED, verbose=-1
    ),
    'CatBoost': CatBoostClassifier(
        iterations=300, learning_rate=0.05,
        depth=6, random_seed=SEED, verbose=0
    )
}
print(f'{len(models)} models defined ✅')

results = {}
trained_pipelines = {}

for name, model in models.items():
    print(f'\nTraining {name} ...')
    pipe = Pipeline([('preprocessor', preprocessor), ('classifier', model)])
    pipe.fit(X_train, y_train)
    
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    
    metrics = {
        'Accuracy' : accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall'   : recall_score(y_test, y_pred, zero_division=0),
        'F1-Score' : f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC'  : roc_auc_score(y_test, y_prob)
    }
    results[name] = metrics
    trained_pipelines[name] = pipe
    
    print(f"  Accuracy: {metrics['Accuracy']:.4f} | F1: {metrics['F1-Score']:.4f} | ROC-AUC: {metrics['ROC-AUC']:.4f}")

print('\n✅ All models trained!')

results_df = pd.DataFrame(results).T.round(4)
results_df = results_df.sort_values('ROC-AUC', ascending=False)
print(results_df.to_string())

metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']

fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(22, 5))
colors = sns.color_palette('muted', len(results_df))

for ax, metric in zip(axes, metrics_to_plot):
    vals = results_df[metric]
    bars = ax.bar(vals.index, vals.values, color=colors, edgecolor='black')
    ax.set_title(metric, fontsize=12)
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis='x', rotation=25)
    for bar, v in zip(bars, vals.values):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.01,
                f'{v:.3f}', ha='center', fontsize=8)

plt.suptitle('Model Comparison', fontsize=15)
plt.tight_layout()
plt.savefig('models/model_comparison.png', bbox_inches='tight')
plt.show()

best_name = results_df['ROC-AUC'].idxmax()
best_pipe = trained_pipelines[best_name]
print(f'Best model: {best_name} (ROC-AUC = {results_df.loc[best_name, "ROC-AUC"]:.4f})')

y_pred_best = best_pipe.predict(X_test)
y_prob_best = best_pipe.predict_proba(X_test)[:, 1]

print(f'Classification Report – {best_name}')
print(classification_report(y_test, y_pred_best,
                             target_names=le.classes_))

                             fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title(f'Confusion Matrix – {best_name}')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob_best)
auc = roc_auc_score(y_test, y_prob_best)
axes[1].plot(fpr, tpr, lw=2, color='darkorange', label=f'AUC = {auc:.4f}')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title(f'ROC Curve – {best_name}')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.savefig('models/roc_curve.png', bbox_inches='tight')
plt.savefig('models/confusion_matrix.png', bbox_inches='tight')
plt.show()

classifier = best_pipe.named_steps['classifier']
prep = best_pipe.named_steps['preprocessor']

try:
    # Get feature names after preprocessing
    num_names = num_features
    cat_names = []
    if cat_features:
        cat_names = prep.named_transformers_['cat']['encoder'].get_feature_names_out(cat_features).tolist()
    all_feature_names = num_names + cat_names

    if hasattr(classifier, 'feature_importances_'):
        importances = classifier.feature_importances_
    elif hasattr(classifier, 'coef_'):
        importances = np.abs(classifier.coef_[0])
    else:
        raise AttributeError('No feature importance available')

    fi_df = pd.DataFrame({'Feature': all_feature_names, 'Importance': importances})
    fi_df = fi_df.sort_values('Importance', ascending=False).head(20)

    plt.figure(figsize=(10, 7))
    sns.barplot(data=fi_df, y='Feature', x='Importance', palette='viridis')
    plt.title(f'Top 20 Feature Importances – {best_name}', fontsize=14)
    plt.tight_layout()
    plt.savefig('models/feature_importance.png', bbox_inches='tight')
    plt.show()

except Exception as e:
    print(f'Feature importance not available: {e}')

    print(f'Running 5-Fold Stratified CV on {best_name} ...')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_scores = cross_val_score(best_pipe, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)

print(f'CV ROC-AUC: {cv_scores.round(4)}')
print(f'Mean: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

plt.figure(figsize=(8, 4))
plt.bar(range(1, 6), cv_scores, color='steelblue', edgecolor='black')
plt.axhline(cv_scores.mean(), color='red', linestyle='--', label=f'Mean = {cv_scores.mean():.4f}')
plt.title('5-Fold Cross-Validation ROC-AUC Scores')
plt.xlabel('Fold')
plt.ylabel('ROC-AUC')
plt.ylim(0.5, 1.0)
plt.legend()
plt.tight_layout()
plt.show()

# Save the best pipeline (includes preprocessor + model)
joblib.dump(best_pipe, 'models/placement_prediction_model.pkl')
joblib.dump(preprocessor, 'models/preprocessor.pkl')
joblib.dump(le, 'models/label_encoder.pkl')

# Save metadata
metadata = {
    'best_model_name'    : best_name,
    'num_features'       : num_features,
    'cat_features'       : cat_features,
    'target_classes'     : list(le.classes_),
    'test_roc_auc'       : float(results_df.loc[best_name, 'ROC-AUC']),
    'test_accuracy'      : float(results_df.loc[best_name, 'Accuracy']),
    'cv_roc_auc_mean'    : float(cv_scores.mean()),
    'cv_roc_auc_std'     : float(cv_scores.std()),
    'all_model_results'  : results
}

with open('models/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=4)

print('Model saved  → models/placement_prediction_model.pkl ✅')
print('Preprocessor → models/preprocessor.pkl ✅')
print('Metadata     → models/model_metadata.json ✅')

print('='*55)
print('          TRAINING SUMMARY')
print('='*55)
print(f'Best Model    : {best_name}')
print(f'Test ROC-AUC  : {results_df.loc[best_name, "ROC-AUC"]:.4f}')
print(f'Test Accuracy : {results_df.loc[best_name, "Accuracy"]:.4f}')
print(f'CV ROC-AUC    : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print('='*55)
print('\nAll Model Leaderboard (by ROC-AUC):')
print(results_df[['Accuracy','F1-Score','ROC-AUC']].to_string())
print('\n✅ Run streamlit_app.py to use the model interactively!')


